# Build ETCCDI zarr store

Reads all per-member, per-scenario NetCDF index files from
`s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/{model}/{scenario}/{member}/{INDEX}.nc`
and writes them into a single consolidated zarr store at
`s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual.zarr`.

## Zarr store structure
```
ETCCDI_indices_annual.zarr/
  {model}/{scenario}/           ← one zarr group per model × scenario
    SU, ID, TXX, TXN, ...       ← one variable per ETCCDI index
    dims: (member, year, lat, lon)
```

## Memory usage
Processing is done one `(model, scenario)` group at a time.  
Peak memory per group (largest case: CESM2-WACCM/SSP2-4.5 with 5 members):
5 members × 16 indices × 50 years × 192 × 288 × 8 bytes ≈ **2.2 GB** — well within 30 GB.

## Expected runtime
~**5–10 minutes** total:  
reading ~300 small nc files (~7 GB uncompressed) from S3 + zarr compression (zstd level 3) + write.

In [1]:
import xarray as xr
import numpy as np
import os
import gc
import shutil
from pathlib import Path
import zarr
import s3fs

In [2]:
# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

# S3 source path (no trailing slash, no 's3://' prefix — used with fs.ls/fs.exists)
INPUT_ROOT = 'reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual'

# S3 destination for the zarr store
ZARR_STORE = 's3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual.zarr'

# Models to include. Missing index files are skipped automatically,
# so partially-complete models (e.g. E3SMv3) are handled gracefully.
MODELS = ['CESM2-WACCM', 'UKESM1-1', 'MIROC-ES2H', 'E3SMv3']

# All 16 ETCCDI index filenames (= variable names in the zarr store)
TEMP_INDICES = ['SU', 'ID', 'TXX', 'TXN', 'FD', 'TN', 'TNX', 'TNN']
PREC_INDICES = ['PRCPTOT', 'SDII', 'RX1D', 'RX5D', 'CDD', 'CWD', 'R10MM', 'R20MM']
ALL_INDICES  = TEMP_INDICES + PREC_INDICES

# Set OVERWRITE=True to delete and recreate the zarr store from scratch.
# Set OVERWRITE=False to skip groups that already exist (resume a partial run).
OVERWRITE = True

In [3]:
# -------------------------------------------------------
# S3 filesystem (authenticated)
# -------------------------------------------------------

fs = s3fs.S3FileSystem(anon=False)

In [4]:
# -------------------------------------------------------
# Helper: read and normalise one index file
# -------------------------------------------------------

_time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

def read_index(fpath, idx_name):
    """
    Open {idx_name}.nc from S3, normalise coordinates, convert time to integer year,
    and return a DataArray named idx_name with dims (year, lat, lon).

    Handles:
    - latitude/longitude  → lat/lon rename (UKESM alistairduffey files)
    - spurious 'height' coordinate (UKESM HiLLA files)
    - cftime and datetime64 time values

    No year filtering is applied here — the full time range of each file is
    preserved so that SSP245 baselines are not truncated.
    """
    ds = xr.open_dataset(
        fpath,
        decode_times=_time_coder,
        engine='h5netcdf',
        storage_options={'anon': False},
    )

    # Rename non-standard spatial coordinates
    rename = {}
    if 'latitude'  in ds.coords: rename['latitude']  = 'lat'
    if 'longitude' in ds.coords: rename['longitude'] = 'lon'
    if rename:
        ds = ds.rename(rename)

    # Drop auxiliary coordinates that vary across models (height, etc.)
    drop = [c for c in ds.coords if c not in ('lat', 'lon', 'time')]
    if drop:
        ds = ds.drop_vars(drop)

    # Extract the single data variable and give it the index name
    data_var = list(ds.data_vars)[0]
    da = ds[data_var].rename(idx_name)

    # Convert time → integer year
    da = da.assign_coords(time=da.time.dt.year.values.astype(int))
    da = da.rename({'time': 'year'})

    return da

In [5]:
# -------------------------------------------------------
# Optionally wipe the existing store
# -------------------------------------------------------

if OVERWRITE and fs.exists(ZARR_STORE):
    fs.rm(ZARR_STORE, recursive=True)
    print(f'Removed existing store: {ZARR_STORE}')

Removed existing store: s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual.zarr


In [6]:
# -------------------------------------------------------
# Main build loop
# -------------------------------------------------------

import time as _time
t0 = _time.time()

for model in MODELS:
    model_prefix = f'{INPUT_ROOT}/{model}'
    if not fs.exists(model_prefix):
        print(f'[SKIP] {model}: directory not found')
        continue

    for scenario_prefix in sorted(fs.ls(model_prefix, detail=False)):
        scenario = scenario_prefix.split('/')[-1]
        if scenario.startswith('.') or not fs.isdir(scenario_prefix):
            continue
        zarr_group = f'{model}/{scenario}'

        # Skip if group already written and OVERWRITE is False
        if not OVERWRITE and fs.exists(ZARR_STORE):
            root = zarr.open(ZARR_STORE, mode='r', storage_options={'anon': False})
            if zarr_group in root:
                print(f'[SKIP] {zarr_group}: already in store')
                continue

        # Discover members (ignore hidden dirs)
        members = sorted(
            p.split('/')[-1]
            for p in fs.ls(scenario_prefix, detail=False)
            if not p.split('/')[-1].startswith('.') and fs.isdir(p)
        )
        if not members:
            print(f'[SKIP] {zarr_group}: no member directories')
            continue

        print(f'Processing {zarr_group}  members={members}')

        # --- Build Dataset: dims = (member, year, lat, lon) ---
        member_datasets = []
        for member in members:
            member_prefix = f'{scenario_prefix}/{member}'
            index_arrays = {}
            for idx in ALL_INDICES:
                s3_key = f'{member_prefix}/{idx}.nc'
                if fs.exists(s3_key):
                    index_arrays[idx] = read_index(f's3://{s3_key}', idx)
                else:
                    print(f'  [WARN] missing: {member}/{idx}.nc')

            if index_arrays:
                ds_member = xr.Dataset(index_arrays)
                ds_member = ds_member.expand_dims({'member': [member]})
                member_datasets.append(ds_member)

        if not member_datasets:
            print(f'  [SKIP] no data loaded')
            continue

        combined = xr.concat(member_datasets, dim='member')

        # Chunk: one member per chunk, full time and spatial extent.
        combined = combined.chunk({'member': 1, 'year': -1, 'lat': -1, 'lon': -1})

        # Write to S3 zarr store
        combined.to_zarr(ZARR_STORE, group=zarr_group, mode='a',
                         storage_options={'anon': False})

        print(f'  -> written ({combined.nbytes / 1e6:.0f} MB uncompressed)')

        del combined, member_datasets
        gc.collect()

elapsed = _time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min. Zarr store: {ZARR_STORE}')

Processing CESM2-WACCM/G6-1.5K-HiLLA  members=['r1', 'r2', 'r3']


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fefddfd7110>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7fefddf9b710>, 14557.713011651)])']
connector: <aiohttp.connector.TCPConnector object at 0x7fefde14ae40>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fefdea50fe0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7fefde9c3a70>, 14557.728024454)])']
connector: <aiohttp.connector.TCPConnector object at 0x7fefdfdd67e0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fefde9ef980>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7fefde9c3350>, 14557.748090686)])']
connector: <aiohttp.connector.TCPConnector object at 0x7fefdeb42cc0>
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is 

  -> written (846 MB uncompressed)
Processing CESM2-WACCM/G6-1.5K-SAI  members=['r1', 'r2', 'r3']


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/core/array.py:3954: UserWarning: The dtype `StringDType()` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  result = await AsyncArray._create_v3(
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/api/asynchronous.py:197: UserWarning: Consolida

  -> written (846 MB uncompressed)
Processing CESM2-WACCM/SSP2-4.5  members=['r10', 'r6', 'r7', 'r8', 'r9']


...

  -> written (700 MB uncompressed)
Processing E3SMv3/SSP2-4.5  members=['r1', 'r2', 'r3']


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/core/array.py:3954: UserWarning: The dtype `StringDType()` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  result = await AsyncArray._create_v3(
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/api/asynchronous.py:197: UserWarning: Consolida

  -> written (1089 MB uncompressed)
Processing E3SMv3/SSP245  members=['r1', 'r2', 'r3']


/tmp/ipykernel_2549/3069307855.py:66: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  combined.to_zarr(ZARR_STORE, group=zarr_group, mode='a',
/tmp/ipykernel_2549/3069307855.py:66: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  combined.to_zarr(ZARR_STORE, group=zarr_group, mode='a',
/tmp/ipykernel_2549/3069307855.py:66: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  combined.to_zarr(ZARR_STORE, group=zarr_group, mode='a',
/tmp/ipykernel_2549/3069307855.py:66: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  combined.to_zarr(ZARR_STORE, group=zarr_group, mode='a',
/tmp/ipykernel_2549/3069307855.py:66: SerializationWarning: saving variable None with floating point dat

  -> written (1380 MB uncompressed)

Done in 4.8 min. Zarr store: s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual.zarr


In [7]:
# -------------------------------------------------------
# Verification: print store tree and open a sample group
# -------------------------------------------------------

root = zarr.open(ZARR_STORE, mode='r', storage_options={'anon': False})
print('=== Zarr store tree ===')
print(root.tree())

# Open one group with xarray to verify it round-trips correctly
sample_model    = MODELS[0]
sample_scenario = next(iter(root[sample_model].keys()))
sample_group    = f'{sample_model}/{sample_scenario}'
print(f'\n=== Sample group: {sample_group} ===')
ds_check = xr.open_zarr(ZARR_STORE, group=sample_group,
                         storage_options={'anon': False}, consolidated=False)
print(ds_check)
print('\nSU year range:', ds_check['SU'].coords['year'].values[[0, -1]])

=== Zarr store tree ===


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vl

/
├── CESM2-WACCM
│   ├── G6-1.5K-HiLLA
│   │   ├── CDD (3, 51, 192, 288) float64
│   │   ├── CWD (3, 51, 192, 288) float64
│   │   ├── FD (3, 51, 192, 288) int64
│   │   ├── ID (3, 51, 192, 288) int64
│   │   ├── PRCPTOT (3, 51, 192, 288) float32
│   │   ├── R10MM (3, 51, 192, 288) int64
│   │   ├── R20MM (3, 51, 192, 288) int64
│   │   ├── RX1D (3, 51, 192, 288) float32
│   │   ├── RX5D (3, 51, 192, 288) float32
│   │   ├── SDII (3, 51, 192, 288) float64
│   │   ├── SU (3, 51, 192, 288) int64
│   │   ├── TN (3, 51, 192, 288) int64
│   │   ├── TNN (3, 51, 192, 288) float32
│   │   ├── TNX (3, 51, 192, 288) float32
│   │   ├── TXN (3, 51, 192, 288) float32
│   │   ├── TXX (3, 51, 192, 288) float32
│   │   ├── lat (192,) float64
│   │   ├── lon (288,) float64
│   │   ├── member (3,) StringDType()
│   │   └── year (51,) int64
│   ├── G6-1.5K-SAI
│   │   ├── CDD (3, 51, 192, 288) float64
│   │   ├── CWD (3, 51, 192, 288) float64
│   │   ├── FD (3, 51, 192, 288) int64
│   │   ├── ID (3, 51, 192, 288) int64
│   │   ├── PRCPTOT (3, 51, 192, 288) float32
│   │   ├── R10MM (3, 51, 192, 288) int64
│   │   ├── R20MM (3, 51, 192, 288) int64
│   │   ├── RX1D (3, 51, 192, 288) float32
│   │   ├── RX5D (3, 51, 192, 288) float32
│   │   ├── SDII (3, 51, 192, 288) float64
│   │   ├── SU (3, 51, 192, 288) int64
│   │   ├── TN (3, 51, 192, 288) int64
│   │   ├── TNN (3, 51, 192, 288) float32
│   │   ├── TNX (3, 51, 192, 288) float32
│   │   ├── TXN (3, 51, 192, 288) float32
│   │   ├── TXX (3, 51, 192, 288) float32
│   │   ├── lat (192,) float64
│   │   ├── lon (288,) float64
│   │   ├── member (3,) StringDType()
│   │   └── year (51,) int64
│   └── SSP2-4.5
│       ├── CDD (5, 86, 192, 288) float64
│       ├── CWD (5, 86, 192, 288) float64
│       ├── FD (5, 86, 192, 288) int64
│       ├── ID (5, 86, 192, 288) int64
│       ├── PRCPTOT (5, 86, 192, 288) float32
│       ├── R10MM (5, 86, 192, 288) int64
│       ├── R20MM (5, 86, 192, 288) int64
│       ├── RX1D (5, 86, 192, 288) float32
│       ├── RX5D (5, 86, 192, 288) float32
│       ├── SDII (5, 86, 192, 288) float64
│       ├── SU (5, 86, 192, 288) int64
│       ├── TN (5, 86, 192, 288) int64
│       ├── TNN (5, 86, 192, 288) float32
│       ├── TNX (5, 86, 192, 288) float32
│       ├── TXN (5, 86, 192, 288) float32
│       ├── TXX (5, 86, 192, 288) float32
│       ├── lat (192,) float64
│       ├── lon (288,) float64
│       ├── member (5,) StringDType()
│       └── year (86,) int64
├── E3SMv3
│   ├── G6-1.5K-HiLLA
│   │   ├── CDD (3, 50, 180, 360) float64
│   │   ├── CWD (3, 50, 180, 360) float64
│   │   ├── FD (3, 50, 180, 360) int64
│   │   ├── ID (3, 50, 180, 360) int64
│   │   ├── PRCPTOT (3, 50, 180, 360) float32
│   │   ├── R10MM (3, 50, 180, 360) int64
│   │   ├── R20MM (3, 50, 180, 360) int64
│   │   ├── RX1D (3, 50, 180, 360) float32
│   │   ├── RX5D (3, 50, 180, 360) float32
│   │   ├── SDII (3, 50, 180, 360) float64
│   │   ├── SU (3, 50, 180, 360) int64
│   │   ├── TN (3, 50, 180, 360) int64
│   │   ├── TNN (3, 50, 180, 360) float32
│   │   ├── TNX (3, 50, 180, 360) float32
│   │   ├── TXN (3, 50, 180, 360) float32
│   │   ├── TXX (3, 50, 180, 360) float32
│   │   ├── lat (180,) float64
│   │   ├── lon (360,) float64
│   │   ├── member (3,) StringDType()
│   │   └── year (50,) int64
│   ├── G6-1.5K-SAI
│   │   ├── CDD (3, 36, 180, 360) float64
│   │   ├── CWD (3, 36, 180, 360) float64
│   │   ├── FD (3, 36, 180, 360) int64
│   │   ├── ID (3, 36, 180, 360) int64
│   │   ├── PRCPTOT (3, 36, 180, 360) float32
│   │   ├── R10MM (3, 36, 180, 360) int64
│   │   ├── R20MM (3, 36, 180, 360) int64
│   │   ├── RX1D (3, 36, 180, 360) float32
│   │   ├── RX5D (3, 36, 180, 360) float32
│   │   ├── SDII (3, 36, 180, 360) float64
│   │   ├── SU (3, 36, 180, 360) int64
│   │   ├── TN (3, 36, 180, 360) int64
│   │   ├── TNN (3, 36, 180, 360) float32
│   │   ├── TNX (3, 36, 180, 360) float32
│   │   ├── TXN (3, 36, 180, 360) float32
│   │   ├── TXX (3, 36, 180, 360) float32
│   │   ├── lat (18



=== Sample group: CESM2-WACCM/SSP2-4.5 ===


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vl

<xarray.Dataset> Size: 2GB
Dimensions:  (member: 5, year: 86, lat: 192, lon: 288)
Coordinates:
  * year     (year) int64 688B 2015 2016 2017 2018 2019 ... 2097 2098 2099 2100
  * member   (member) object 40B 'r10' 'r6' 'r7' 'r8' 'r9'
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
Data variables: (12/16)
    CDD      (member, year, lat, lon) float64 190MB dask.array<chunksize=(1, 86, 192, 288), meta=np.ndarray>
    CWD      (member, year, lat, lon) float64 190MB dask.array<chunksize=(1, 86, 192, 288), meta=np.ndarray>
    R20MM    (member, year, lat, lon) int64 190MB dask.array<chunksize=(1, 86, 192, 288), meta=np.ndarray>
    FD       (member, year, lat, lon) int64 190MB dask.array<chunksize=(1, 86, 192, 288), meta=np.ndarray>
    ID       (member, year, lat, lon) int64 190MB dask.array<chunksize=(1, 86, 192, 288), meta=np.ndarray>
    R10MM    (member, year, lat, lon) int64

In [8]:
# -------------------------------------------------------
# Store size on S3
# -------------------------------------------------------

total_bytes = fs.du(ZARR_STORE)
print(f'Zarr store size on S3: {total_bytes / 1e9:.2f} GB')

Zarr store size on S3: 3.49 GB
